# Visualization - Layouts, styles, faces

Notes based on: https://etetoolkit.github.io/ete/tutorial/tutorial_smartview.html#customizing-the-visualization

For exercises and a tutorial, see: https://github.com/etetoolkit/ete-gallery

---

The main elements used to customize the visualization are *layouts*,
*styles*, and *faces*.

## Layouts

Layouts contain the `draw_node()` and `draw_tree()` functions, which
create the styles and faces that we use to represent the tree. They
are objects of the class
[Layout](https://etetoolkit.github.io/ete/reference/reference_smartview.html#ete4.smartview.layout.Layout).
They contain:

- `name`: Identifies the layout, so it can be activated/deactivated in the GUI.
- `draw_tree()`: A function that produces style and faces for the full tree.
- `draw_node()`: A function that produces style and faces for the given nodes.
- `cache_size`: The number of nodes cached when calling `draw_node` (defaults to all).
- `active`: Whether the layout will be immediately active when exploring (defaults to True).

Let's look at how to use them. The simplest case is:

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout

t = Tree()
t.populate(20)

layout = Layout(name='I am a layout doing nothing')

t.explore(layouts=[layout])

We can see a representation of the tree, and in the control panel, a
layout that appears with the name "I am a layout doing nothing".

![layout example](layout_example.png)

Its name is accurate, as we can see if we activate or desactivate it by
clicking its checkbox: nothing happens, no extra information is shown
in the tree anyway.

## Changing the tree style

The `draw_tree` field of a layout specifies the general aspects of the
tree style. For example, we can modify the scale used to render tree
branches or choose between circular or rectangular tree drawing, etc.

In general, it is a function of the tree, and returns a list of
faces and styles to use.

We often just need to change its style, and in a way that does not
depend on the tree. For that common case, `draw_tree` can also be a
dictionary with the style.

A dictionary with the tree style can look like this:

```python
my_tree_style = {
   'shape': 'circular',  # or 'rectangular'
   'radius': 5,  # in circular mode, minimum radius value
   'angle-start': -pi/2,  # in circular mode, where to start
   'angle-end': pi/2,  # alternatively we can give 'angle-span'
   'node-height-min': 10,  # when to start collapsing nodes
   'content-height-min': 5,  # when to start showing faces
   'collapsed': {'shape': 'outline', 'fill-opacity': 0.8},
   'show-popup-props': None,  # show all defined properties
   'hide-popup-props': ['support'],  # except support
   'is-leaf-fn': lambda node: node.level > 4,  # nodes treated as leaves
   'box': {'fill': 'green', 'opacity': 0.1, 'stroke': 'blue'},
   'dot': {'shape': 'hexagon', 'fill': 'red'},
   'hz-line': {'stroke-width': 2},  # horizontal line to parent
   'vt-line': {'stroke': '#ffff00'},  # vertical line to children
}
```

The last four (**box**, **dot**, **hz-line**, **vt-line**) define the
general look for all the nodes, but they can be overriden too in an
individual basis with the function `draw_node` as explained below.

The
[Layout](https://etetoolkit.github.io/ete/reference/reference_smartview.html#module-ete4.smartview.layout)
documentation has a complete list of options.

Let's see some examples of how to modify the tree style.

### Example of simple change

A simple way to control the tree style is to pass a dictionary with
the options we want to `draw_tree`:

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout

t = Tree()
t.populate(20)

layout = Layout(name='my layout',
                draw_tree={'node-height-min': 100})

t.explore(layouts=[layout])

In this case, we are requesting to collapse any node with a height
less than 100 pixels.

## Changing the node style

In the same way that we can control the general tree style with a
dictionary of options returned by `draw_tree`, we can also control the
style of a given node with a dictionary of options returned by
`draw_node`.

It is possible to change the color, thickness of lines and many more
style attributes of the following node elements:

- **box**: The box (area) surrounding the node.
- **dot**: The dot that represents the node itself.
- **hz-line**: The horizontal line that connects it to its parent.
- **vt-line**: The vertical line connecting it to its children.

For all of them there are many options to change their style. The main
options are **fill**, **stroke**, **stroke-width**, **opacity**, but
there are also many more: **fill-opacity**, **stroke-opacity**,
**stroke-linecap**, **stroke-linejoin**, **stroke-miterlimit**,
**stroke-dashoffset**, **stroke-dasharray**, **paint-order**,
**fill-rule**, etc. They are all based on [SVG
attributes](https://developer.mozilla.org/en-US/docs/Web/SVG/Tutorial/Fills_and_Strokes).

In addition to those, some elements have extra attributes:

- **dot**
  - **shape**: Figure (circle, square, ...) or its number of sides, representing the node.
  - **radius**: The approximate radius in pixels of the dot.
- **collapsed** (only used from the tree style)
  - **shape**: Representation of collapsed nodes as "skeleton" or "outline".

### Example of simple change

A simple tree where we change the style for the leaves:

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout

t = Tree('((a,b),c);')

# Nodes will be represented as small red triangles of 10 pixels radius.
style_dot = {'shape': 'triangle',
             'radius': 10,
             'fill': 'red'}

# Branch lines (horizontal lines) will be brown and dashed, and 10 pixels thick.
style_hz_line = {'stroke-dasharray': '5,5',
                 'stroke-width': 10,
                 'stroke': '#964B00'}

def draw_node(node):
    if node.is_leaf:
        return {'dot': style_dot,
                'hz-line': style_hz_line}

layout = Layout(name='My layout', draw_node=draw_node)
t.explore(layouts=[layout])

![my layout](my_layout.png)

We can use different styles for different nodes. Let's see a couple of
examples:

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout

t = Tree('((a,b),c);')

# Leaves will have small red squares of radius 10 pixels.
leaf_style = {'dot': {'shape': 'square',
                      'radius': 5,
                      'fill': 'red'}}

# The root node will be blue, with a dot of 15 pixel radius, and a
# custom vertical line.
root_style = {'dot': {'radius': 15, 'fill': 'blue'},
              'vt-line': {'stroke': '#964B00',
                          'stroke-width': 10,
                          'stroke-dasharray': '5,5'}}

def draw_node(node):
    if node.is_leaf:
        return leaf_style
    elif node.is_root:
        return root_style

layout = Layout(name='My layout', draw_node=draw_node)
t.explore(layouts=[layout])

![draw node](draw_node.png)

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout, BASIC_LAYOUT

t = Tree('((((a,b),c),d),e);')

vowels = {'a', 'e', 'i', 'o', 'u'}

def draw_node(node):
    if node.is_leaf:
        if node.name in vowels:
            return {'dot': {'radius': 8},
                    'box': {'fill': 'red'}}

layout = Layout(name='My layout', draw_node=draw_node)
t.explore(layouts=[BASIC_LAYOUT, layout])

![vowels](vowels.png)

## Faces

Faces are small pieces of graphical information that can be shown in
nodes. For instance, text labels or external images can be linked to
nodes and they will be plotted within the tree image.

Several types of node faces are provided by the
[ete4.smartview.faces](https://etetoolkit.github.io/ete/reference/reference_smartview.html#module-ete4.smartview.faces)
submodule, ranging from simple text
[TextFace](https://etetoolkit.github.io/ete/reference/reference_smartview.html#ete4.smartview.faces.TextFace)
and geometric shapes
([CircleFace](https://etetoolkit.github.io/ete/reference/reference_smartview.html#ete4.smartview.faces.CircleFace)
or
[RectFace](https://etetoolkit.github.io/ete/reference/reference_smartview.html#ete4.smartview.faces.RectFace)),
to molecular sequence representations ([SeqFace](https://etetoolkit.github.io/ete/reference/reference_smartview.html#ete4.smartview.faces.SeqFace)).

A complete list of available faces can be found at the [faces module
page](https://etetoolkit.github.io/ete/reference/reference_smartview.html#module-ete4.smartview.faces).


### Positioning faces

Faces can be placed on different positions around the node, namely
**top**, **bottom**, **right**, **left**, or **aligned** (and for
texts produced by the function `draw_tree`, also **header**).

For instance, if you want two text labels drawn below the branch line
of a given node, a pair of `TextFace` faces can
be created on the columns 0 and 1 of the **bottom** position, which
are returned by the function `draw_node`:

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout, TextFace

t = Tree('((a:1,b:1):1,c:1):1;')

def draw_node(node):
    if node.is_root:
        return [TextFace('hello', style={'fill': 'red'},
                         column=0, position='bottom'),
                TextFace('world', style={'fill': 'blue'},
                         column=1, position='bottom')]

layout = Layout(name='My layout', draw_node=draw_node)
t.explore(layouts=[layout])

![face bottom](face_bottom.png)

If we set the column of "world" to 0 too:

In [ ]:
from ete4 import Tree
[...]
        return [TextFace('hello', style={'fill': 'red'},
                         column=0, position='bottom'),
                TextFace('world', style={'fill': 'blue'},
                         column=0, position='bottom')]
[...]
t.explore(layouts=[layout])

![face bottom 2](face_bottom2.png)

So if we add more than one face to the same area and column, they will
be piled up.

If the position is **aligned**, the face will be drawn in an aligned
column. Let's see an example:

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout, TextFace

t = Tree('((a,b),c);')

def draw_node(node):
    if node.is_leaf:
        return [TextFace('hello', style={'fill': 'red'},
                         column=0, position='aligned'),
                TextFace('world', style={'fill': 'blue'},
                         column=1, position='aligned')]

layout = Layout(name='My layout', draw_node=draw_node)
t.explore(layouts=[layout])

![face aligned](face_aligned.png)

### Collapsed nodes

When viewing a large tree, ETE will collapse all the branches that are
too small to be seen. When a branch is collapsed, we see a skeleton of
its contents by default (but the shape can be changed to a triangular
outline if we set it in the tree style, `{'collapsed': {'shape':
'outline'}}`).

When drawing collapsed nodes, we see by default the faces that
correspond to all the sibling nodes that are collapsed together. But
we can fine-tune this behavior by passing a second argument to the
`draw_node` function. This argument will contain a list of all the
collapsed (sibling) nodes at the time of the drawing, and it will be
empty if the node is not collapsed. The function will look like
`draw_node(node, collapsed)`:

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout, TextFace

t = Tree('((a:1,b:1)n1:1,c):1;', parser='name')

def draw_node(node, collapsed):
    if node.name == 'n1':
        if collapsed:
            return TextFace('n1 is collapsed', column=0, position='right')
        else:
            return TextFace('n1 is NOT collapsed', column=0, position='right')

layout = Layout(name='My layout', draw_node=draw_node)
t.explore(layouts=[layout])

Depending on the zoom level, we now will see:

![not collapsed](not_collapsed.png)
![collapsed](collapsed.png)


### Face properties

Each face has its own properties that control the details of its
drawing. We can find a complete list for each face in the [faces
module
documentation](https://etetoolkit.github.io/ete/reference/reference_smartview.html#module-ete4.smartview.faces).

Let's look at a simple example:

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout, TextFace

t = Tree('((a:1,b:1):1,c:1):1;')

# Create a TextFace initialized with certain properties.
face_top = TextFace('branch top!',
                    fs_min=6, fs_max=25, rotation=-10,
                    style={'fill': 'blue', 'font-family': 'courier'},
                    position='top', anchor=(-1, 1))

# Same thing, but adding them after initializing it.
face_bottom = TextFace('branch bottom!')
face_bottom.fs_min = 6
face_bottom.fs_max = 25
face_bottom.rotation = 10
face_bottom.style = {'fill': 'red', 'font-family': 'sans-serif'}
face_bottom.position = 'bottom'
face_bottom.anchor = (-1, -1)

def draw_node(node):
    return [face_top, face_bottom]

layout = Layout(name='My layout', draw_node=draw_node)
t.explore(layouts=[layout])

![face properties](face_properties.png)


#### Combining styles and faces

The `draw_node` function can return a list with as many styles and
faces as we want.

If it returns just one element (a face or a dictionary
representing a style), it is intepreted as a list of only one element
(itself).

Instead of returning a list, we can also
[yield](https://docs.python.org/3/reference/expressions.html#yieldexpr)
elements, that is, `draw_node` would become a [python
generator](https://peps.python.org/pep-0255/). This syntax is the
cleanest and this way is the most efficient of all, so we will use it
in the following examples.

Let's see an example combining styles and faces, for both the tree
(with `draw_tree`) and the nodes (with `draw_node`):

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout, TextFace, LegendFace

t = Tree('((((a,b),c),d),e);')

def draw_tree(tree):
    yield {'dot': {'opacity': 1, 'fill': 'black'}}

    yield TextFace('Vowel?', fs_min=6, fs_max=16, position='header')

    yield LegendFace('Type of letter',
                     variable='discrete',
                     colormap={'vowel': 'red', 'consonant': 'blue'})

vowels = {'a', 'e', 'i', 'o', 'u'}

def draw_node(node):
    if not node.is_leaf:
        return

    yield {'dot': {'shape': 'triangle', 'radius': 8}}

    if node.name in vowels:
        yield {'box': {'fill': 'red'}}
        yield TextFace('yes, a vowel', style={'fill': 'red'},
                       position='aligned')
    else:
        yield {'box': {'fill': 'blue'}}
        yield TextFace('not a vowel', style={'fill': 'blue'},
                       position='aligned')

layout = Layout('Vowels layout', draw_tree=draw_tree, draw_node=draw_node)
t.explore(layouts=[layout])

![combined](combined.png)

Let's see another example, where we change the background of certain
nodes that are common ancestors to other nodes we are interested in:

In [ ]:
from ete4 import Tree
from ete4.smartview import Layout, TextFace

t = Tree('((((a1,a2),a3), ((b1,b2),(b3,b4))), ((c1,c2),c3));')

# Background colors.
style1 = {'box': {'fill': 'LightSteelBlue'}}
style2 = {'box': {'fill': 'Moccasin'}}
style3 = {'box': {'fill': 'DarkSeaGreen'}}
style4 = {'box': {'fill': 'Khaki'}}

# Find common ancestors.
n1 = t.common_ancestor(['a1', 'a2', 'a3'])
n2 = t.common_ancestor(['b1', 'b2', 'b3', 'b4'])
n3 = t.common_ancestor(['c1', 'c2', 'c3'])
n4 = t.common_ancestor(['b3', 'b4'])

def draw_node(node):
    # Add node name with big text.
    yield TextFace(node.name, fs_min=6, fs_max=25, position='right')

    # Set the node style.
    if node == n1:
        yield style1
    elif node == n2:
        yield style2
    elif node == n3:
        yield style3
    elif node == n4:
        yield style4

layout = Layout('My layout', draw_node=draw_node)
t.explore(layouts=[layout])

![node backgrounds](node_backgrounds.png)